In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import logrank

# Log-Rank Test for Freezing-Temperature Distributions

This notebook uses a two-sided log-rank test to compare two sets of droplet-freezing temperatures.

The test is most appropriate when the **expected difference** resembles a general shift towards warmer or colder freezing. It may have low power when curves cross, have different shapes, or contain different ice-active populations. In these cases, the Integrated Squared Difference (ISD) test may be more appropriate.

A small p-value provides evidence that the two underlying freezing-temperature distributions d However, the test does not show where or why they differ. p = 0.05 is often used as the threshold for statistical significance.iffer. A non-significant result does not prove that the distributions are identical, particularly when droplet numbers are small.

## Input

The CSV file should contain two columns of freezing temperatures, with the group names as column headings. Blank and non-numeric cells are removed automatically.

## Use

1. Place the CSV in the same folder as the notebook.
2. Enter its filename in the **Setup** cell.
3. Run all cells in order.
4. Check the group names and droplet numbers.
5. Inspect both the p-value and the fraction-frozen plot.ction-frozen plot.of droplets in each group should therefore always be reported alongside the test result.means or median freezing temperatures differ.


In [ ]:
# ========================================================================
# Setup
# ========================================================================

CSV_FILE = "example_data_tests.csv"
OUTPUT_FILE = "logrank_test_results.csv"

TEMP_INCREMENT = 0.01

FIGURE_FILE = "logrank_fraction_frozen.png"
FIGURE_DPI = 300

In [ ]:
# ========================================================================
# Calculate fraction frozen
# ========================================================================

def fraction_frozen(temps, t_grid):
    """Calculate fraction frozen on a temperature grid."""

    temps = np.sort(temps)

    counts = np.searchsorted(
        temps,
        t_grid,
        side="right")

    return (len(temps) - counts) / len(temps)

In [ ]:
# ============================================================
# Calculate logrank statistic with scipy.logrank 
# ============================================================
def logrank_hypothesis_test(temps1, temps2):
    """Perform a two-sided log-rank test."""

    # Change the sign so duration increases as temperature falls
    result = logrank(
        -temps1,
        -temps2,
        alternative="two-sided")

    # SciPy returns a Z-statistic.
    # Squaring it gives the usual chi-squared log-rank statistic.
    test_statistic = result.statistic ** 2
    p_value = result.pvalue

    print(f"Observed log-rank statistic: {test_statistic:.4f}")
    print(f"p-value: {p_value:.4f}")

    return test_statistic, p_value

In [ ]:
# ============================================================
# Plotting stuff
# NOTE: largely written by MS Copilot
# ============================================================

def plot_fraction_frozen(temps1, temps2, p_value):

    # Common temperature grid
    t_min = min(temps1.min(), temps2.min())
    t_max = max(temps1.max(), temps2.max())

    t_grid = np.arange(
        t_min - TEMP_INCREMENT,
        t_max + TEMP_INCREMENT,
        TEMP_INCREMENT)

    # Fraction-frozen curves
    ff1 = fraction_frozen(temps1, t_grid)
    ff2 = fraction_frozen(temps2, t_grid)

    # Plot
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.plot(t_grid, ff1, label=COLUMN_1)
    ax.plot(t_grid, ff2, label=COLUMN_2)

    ax.set_xlabel("Temperature (°C)")
    ax.set_ylabel("Fraction frozen")
    ax.set_title("Fraction-frozen curves")
    ax.set_ylim(0, 1)
    ax.legend()

    # Add p-value inside the figure
    ax.text(
        0.7,
        0.8,
        f"Log-rank p = {p_value:.4f}",
        transform=ax.transAxes,
        verticalalignment="top")

    fig.tight_layout()

    fig.savefig(
        FIGURE_FILE,
        dpi=FIGURE_DPI,
        bbox_inches="tight")

    print(f"Figure saved as: {FIGURE_FILE}")

    plt.show()

In [ ]:
# ============================================================
# Read the CSV file
# ============================================================

# Read the CSV file
data = pd.read_csv(CSV_FILE)

# Use the first two columns
COLUMN_1 = data.columns[0]
COLUMN_2 = data.columns[1]

# Convert values to numbers and remove blanks
temps1 = pd.to_numeric(
    data[COLUMN_1],
    errors="coerce"
).dropna().to_numpy()

temps2 = pd.to_numeric(
    data[COLUMN_2],
    errors="coerce"
).dropna().to_numpy()

print(f"Comparing: {COLUMN_1} and {COLUMN_2}")
print(f"{COLUMN_1}: {len(temps1)} observations")
print(f"{COLUMN_2}: {len(temps2)} observations")

In [ ]:
# ============================================================
# Run the test and make the plot
# ============================================================

test_statistic, p_value = logrank_hypothesis_test(
    temps1,
    temps2)


plot_fraction_frozen(
    temps1,
    temps2,
    p_value)
